In [27]:
import os
import librosa
import numpy as np
import pandas as pd
from tqdm import tqdm


In [28]:
folders = [
# r"../Data/raw/saxophone_data",
# r"../Data/raw/piano_data",
# r"../Data/raw/guitar_data",
r"../Data/raw/violin_data"
]

In [29]:
def extract_features(y, sr):
    # MFCC
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
    mfcc_mean = np.mean(mfcc, axis=1)

    # Chroma
    chroma = librosa.feature.chroma_stft(y=y, sr=sr)
    chroma_mean = np.mean(chroma, axis=1)

    # Spectral features
    centroid = np.mean(librosa.feature.spectral_centroid(y=y, sr=sr))
    bandwidth = np.mean(librosa.feature.spectral_bandwidth(y=y, sr=sr))
    rolloff = np.mean(librosa.feature.spectral_rolloff(y=y, sr=sr))

    # ZCR
    zcr = np.mean(librosa.feature.zero_crossing_rate(y))

    # Tempo
    tempo, _ = librosa.beat.beat_track(y=y, sr=sr)
    # Xử lý tương thích: librosa bản mới trả về tempo là mảng 1D thay vì số float
    tempo_val = tempo[0] if isinstance(tempo, np.ndarray) else tempo

    features = np.hstack([
        mfcc_mean,
        chroma_mean,
        centroid,
        bandwidth,
        rolloff,
        zcr,
        tempo_val
    ])

    return features

In [30]:
data = []
# --- CẤU HÌNH SLIDING WINDOW ---
window_duration = 5.0  # Độ dài mỗi cửa sổ là 5 giây
overlap_ratio = 0.5    # Chồng lấp 50%
hop_duration = window_duration * (1 - overlap_ratio) # Bước nhảy là 2.5 giây

for folder in folders:
    instrument = os.path.basename(folder)

    for file in tqdm(os.listdir(folder)):
        if file.endswith(".mp3"):
            path = os.path.join(folder, file)

            try:
                # 1. Đọc file âm thanh
                y, sr = librosa.load(path, sr=22050)
                
                # 2. Quy đổi thời gian ra số lượng sample (mẫu)
                window_samples = int(window_duration * sr)
                hop_samples = int(hop_duration * sr)

                # 3. Logic Sliding Window
                start_sample = 0
                segment_id = 1
                
                # Bỏ qua các file có tổng thời lượng ngắn hơn 1 cửa sổ
                if len(y) < window_samples:
                    print(f"Bỏ qua {file} vì thời lượng quá ngắn.")
                    continue

                # Trượt cửa sổ cho đến khi cửa sổ vượt quá chiều dài của file
                while start_sample + window_samples <= len(y):
                    end_sample = start_sample + window_samples
                    y_segment = y[start_sample:end_sample]

                    # Trích xuất đặc trưng trên đoạn cửa sổ hiện tại
                    features = extract_features(y_segment, sr)

                    # Ghi nhận dữ liệu
                    row = {
                        "file_name": file,
                        "path": path,
                        "instrument": instrument,
                        "segment_id": segment_id
                    }

                    # MFCC (1-13)
                    for i in range(13):
                        row[f"mfcc_{i+1}"] = features[i]

                    # Chroma
                    chroma_labels = [
                        "C","Csharp","D","Dsharp","E","F",
                        "Fsharp","G","Gsharp","A","Asharp","B"
                    ]
                    for i, label in enumerate(chroma_labels):
                        row[f"chroma_{label}"] = features[13 + i]

                    # Các đặc trưng khác
                    row["spectral_centroid"] = features[25]
                    row["spectral_bandwidth"] = features[26]
                    row["spectral_rolloff"] = features[27]
                    row["zero_crossing_rate"] = features[28]
                    row["tempo"] = features[29]

                    data.append(row)
                    
                    # Tiến cửa sổ lên một bước nhảy (hop)
                    start_sample += hop_samples
                    segment_id += 1

            except Exception as e:
                print(f"Error processing {file}: {e}")

df = pd.DataFrame(data)

100%|██████████| 189/189 [06:52<00:00,  2.18s/it]


In [31]:
df

,file_name,path,instrument,segment_id,mfcc_1,mfcc_2,mfcc_3,mfcc_4,mfcc_5,mfcc_6,...,chroma_G,chroma_Gsharp,chroma_A,chroma_Asharp,chroma_B,spectral_centroid,spectral_bandwidth,spectral_rolloff,zero_crossing_rate,tempo
0,vio_001_classic.mp3,../Data/raw/violin_data\vio_001_classic.mp3,violin_data,1,-428.341095,199.171814,58.524216,8.056131,10.465203,-0.851486,...,0.296273,0.143625,0.275417,0.106872,0.155957,618.436980,552.536693,951.498413,0.039908,258.398438
1,vio_001_classic.mp3,../Data/raw/violin_data\vio_001_classic.mp3,violin_data,2,-431.838013,201.575089,71.627411,13.579899,7.982882,-1.503316,...,0.384414,0.114843,0.102661,0.090335,0.245478,443.363632,439.850949,644.301351,0.037250,287.109375
2,vio_001_classic.mp3,../Data/raw/violin_data\vio_001_classic.mp3,violin_data,3,-421.203766,201.097916,63.846626,19.605070,8.377815,-1.074361,...,0.267736,0.137667,0.196846,0.083975,0.173392,458.658234,540.661182,635.927327,0.034905,161.499023
3,vio_001_classic.mp3,../Data/raw/violin_data\vio_001_classic.mp3,violin_data,4,-320.436066,149.970825,52.138680,30.312635,4.046335,1.476054,...,0.337430,0.169770,0.145665,0.084205,0.182908,1328.367666,1618.042326,2625.505575,0.063687,161.499023
4,vio_001_classic.mp3,../Data/raw/violin_data\vio_001_classic.mp3,violin_data,5,-235.463074,102.652794,50.090900,28.749683,3.127921,-1.506148,...,0.379199,0.190269,0.113119,0.097722,0.298499,2211.340063,2767.539886,5094.895426,0.082617,129.199219
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2074,vio_189_classic.mp3,../Data/raw/violin_data\vio_189_classic.mp3,violin_data,7,-325.684540,158.075912,16.051104,33.324245,11.242268,4.558515,...,0.131598,0.142704,0.344986,0.290347,0.386921,875.460308,1497.971080,1335.008748,0.033662,135.999178
2075,vio_189_classic.mp3,../Data/raw/violin_data\vio_189_classic.mp3,violin_data,8,-325.941925,161.968658,20.127333,33.455826,11.636805,3.296699,...,0.306830,0.236642,0.378953,0.170772,0.147274,815.234128,1435.481104,1147.291056,0.033092,143.554688
2076,vio_189_classic.mp3,../Data/raw/violin_data\vio_189_classic.mp3,violin_data,9,-330.017151,164.612564,23.487627,34.389214,11.953078,5.676610,...,0.412183,0.211186,0.247937,0.141109,0.283368,801.275972,1429.136037,1132.835897,0.032918,135.999178
2077,vio_189_classic.mp3,../Data/raw/violin_data\vio_189_classic.mp3,violin_data,10,-332.068146,166.328064,26.130079,36.173122,14.546709,8.534752,...,0.295647,0.197594,0.334935,0.147048,0.227760,741.237609,1391.744780,1017.992147,0.028888,92.285156


In [32]:

df.to_csv("../Results/violin_feature_database.csv", index=False)

print("Feature extraction finished!")
print("Total files processed:", len(df))

Feature extraction finished!
Total files processed: 2079
